# 01 — Build and verify the matched dataset

This notebook pins the environment, checks HF/J-Lens agreement, constructs all 118 reciprocal prompt pairs, selects the WRITTEN layer and threshold using calibration groups only, and freezes the 77 verified held-out pairs. Calibration-only causal interchange is injected explicitly; no causal result is placed in the sanitized manifest consumed by cheap READ.

**Outputs:** `artifacts/final/01_dataset.json`, `01_clean_manifest.json`, and `01_directions.pt`.

In [1]:
from __future__ import annotations

import hashlib
import json
import os
import shutil
import subprocess
import sys
from pathlib import Path

import torch

ROOT = Path.cwd().resolve()
if ROOT.name == 'notebooks':
    ROOT = ROOT.parent
os.chdir(ROOT)
sys.path.insert(0, str(ROOT))
ARTIFACTS = ROOT / 'artifacts' / 'final'
ARTIFACTS.mkdir(parents=True, exist_ok=True)

from src.causal_read import (
    clean_state_and_logits,
    symmetric_interchange,
    token_difference_metric,
)
from src.datasets import build_and_verify_dataset
from src.jlens_interface import (
    MODEL_ID,
    load_model,
    load_published_lens,
    release_model,
)

def write_json(path: Path, value: object) -> None:
    path.parent.mkdir(parents=True, exist_ok=True)
    path.write_text(json.dumps(value, indent=2, sort_keys=True) + '\n')

def sha256(path: Path) -> str:
    return hashlib.sha256(path.read_bytes()).hexdigest()

def progress(event: str, payload: dict) -> None:
    print(event, json.dumps(payload, sort_keys=True))

hf = shutil.which('hf')
if hf is None:
    raise RuntimeError('hf CLI is required')
preflight = {
    'hf_path': hf,
    'hf_identity': subprocess.run([hf, 'auth', 'whoami'], check=True, capture_output=True, text=True).stdout.strip(),
    'gpu_csv': subprocess.run(['nvidia-smi', '--query-gpu=memory.total,memory.free', '--format=csv'], check=True, capture_output=True, text=True).stdout.strip(),
    'torch_version': torch.__version__,
}
print(json.dumps(preflight, indent=2))

bundle = load_model(MODEL_ID)
published_lens = load_published_lens(MODEL_ID)
try:
    built = build_and_verify_dataset(
        bundle,
        published_lens,
        clean_state_and_logits=clean_state_and_logits,
        symmetric_interchange=symmetric_interchange,
        token_difference_metric=token_difference_metric,
        progress=progress,
    )
finally:
    del published_lens
    del bundle
    release_model()

direction_path = ARTIFACTS / '01_directions.pt'
torch.save(built['direction_cache'], direction_path)
direction_record = {
    'path': str(direction_path.relative_to(ROOT)),
    'bytes': direction_path.stat().st_size,
    'sha256': sha256(direction_path),
}
clean_manifest = built['sanitized_manifest']
clean_manifest['direction_cache'].update(direction_record)
clean_path = ARTIFACTS / '01_clean_manifest.json'
write_json(clean_path, clean_manifest)

dataset = built['full_dataset_artifact']
dataset['preflight'] = preflight
dataset['direction_cache'].update(direction_record)
dataset['clean_read_manifest'].update({
    'path': str(clean_path.relative_to(ROOT)),
    'bytes': clean_path.stat().st_size,
    'sha256': sha256(clean_path),
})
dataset_path = ARTIFACTS / '01_dataset.json'
write_json(dataset_path, dataset)

summary = {
    'selection': dataset['selection']['layer'],
    'written_threshold': dataset['selection']['written_threshold'],
    'counts': dataset['counts'],
    'kl_max_mean': dataset['logit_agreement']['max_mean_kl'],
    'dataset_sha256': sha256(dataset_path),
    'clean_manifest_sha256': sha256(clean_path),
    'direction_cache_sha256': sha256(direction_path),
}
print(json.dumps(summary, indent=2))


{
  "hf_path": "/home/jovyan/.local/bin/hf",
  "hf_identity": "user=sushmanth orgs=sushmanthreddy,devoworm-group,OWG,syscv-community,context-course",
  "gpu_csv": "memory.total [MiB], memory.free [MiB]\n143771 MiB, 143072 MiB",
  "torch_version": "2.5.1+cu124"
}


Loading weights:   0%|          | 0/339 [00:00<?, ?it/s]

/home/jovyan/j-space-thoughts/.venv/lib/python3.11/site-packages/transformers/models/qwen2/modeling_qwen2.py:108: UserWarning: Deterministic behavior was enabled with either `torch.use_deterministic_algorithms(True)` or `at::Context::setDeterministicAlgorithms(true)`, but this operation is not deterministic because it uses CuBLAS and you have CUDA >= 10.2. To enable deterministic behavior in this case, you must set an environment variable before running your PyTorch application: CUBLAS_WORKSPACE_CONFIG=:4096:8 or CUBLAS_WORKSPACE_CONFIG=:16:8. For more information, go to https://docs.nvidia.com/cuda/cublas/index.html#results-reproducibility (Triggered internally at ../aten/src/ATen/Context.cpp:208.)
  freqs = (inv_freq_expanded.float() @ position_ids_expanded.float()).transpose(1, 2)
/opt/conda/lib/python3.11/site-packages/torch/nn/modules/linear.py:125: UserWarning: Deterministic behavior was enabled with either `torch.use_deterministic_algorithms(True)` or `at::Context::setDeterminis

kl_gate {"max_mean_kl": 1.6602172081547906e-08, "n": 20, "status": "PASS"}
candidates {"n_calibration_pairs": 25, "n_candidates": 118, "n_evaluation_pairs": 93, "n_tokenization_rejections": 0, "n_tokenized": 118}


/home/jovyan/j-space-thoughts/src/jlens_interface.py:401: UserWarning: Deterministic behavior was enabled with either `torch.use_deterministic_algorithms(True)` or `at::Context::setDeterministicAlgorithms(true)`, but this operation is not deterministic because it uses CuBLAS and you have CUDA >= 10.2. To enable deterministic behavior in this case, you must set an environment variable before running your PyTorch application: CUBLAS_WORKSPACE_CONFIG=:4096:8 or CUBLAS_WORKSPACE_CONFIG=:16:8. For more information, go to https://docs.nvidia.com/cuda/cublas/index.html#results-reproducibility (Triggered internally at ../aten/src/ATen/Context.cpp:208.)
  directions = F.normalize(rows @ jacobian, dim=-1)


...

calibration_layer {"causal_separation": 0.0023909927181904513, "dashboard_abs_C_median": 0.006912442396313364, "engine_abs_C_median": 0.009303435114503815, "layer": 23, "n_pairs": 23}


calibration_layer {"causal_separation": 0.0002231781272650146, "dashboard_abs_C_median": 0.0042198562776983245, "engine_abs_C_median": 0.004443034404963339, "layer": 24, "n_pairs": 22}


calibration_layer {"causal_separation": 0.003095370073226946, "dashboard_abs_C_median": 0.0022026431718061676, "engine_abs_C_median": 0.0052980132450331134, "layer": 25, "n_pairs": 21}


calibration_layer {"causal_separation": 0.002780413240565043, "dashboard_abs_C_median": 0.0032679738562091504, "engine_abs_C_median": 0.006048387096774193, "layer": 26, "n_pairs": 21}
selection {"causal_separation": 0.9265183176859295, "layer": 16, "written_threshold": 2.482430934906006}
verification {"calibration_pairs": 25, "candidates": 118, "engine_verified_evaluation_pairs": 77, "evaluation_pairs": 93, "unverified_pairs": 16, "verified_pairs": 77}


{
  "selection": 16,
  "written_threshold": 2.482430934906006,
  "counts": {
    "candidates": 118,
    "calibration_pairs": 25,
    "evaluation_pairs": 93,
    "verified_pairs": 77,
    "unverified_pairs": 16,
    "engine_verified_evaluation_pairs": 77
  },
  "kl_max_mean": 1.6602172081547906e-08,
  "dataset_sha256": "792856e3907b87ad8e5e69589767c2c85fcd6df45eaf22dfdd2a594ea6215eb8",
  "clean_manifest_sha256": "41d5ca9f014dd73b05f63e458915c6edcfe6a0449fe022469494471daa9a5bb5",
  "direction_cache_sha256": "7a00eac9247d2c7160ae0c9ac49e6043201b4e86b493c1cb934b37a27a1e0b12"
}
